# E8-C Colab — E7-C(Qwen3.5-27B LoRA) + Order4 · Gate · Zoom · Blend (A100 80GB)

**재학습 없이** E7-C 어댑터에 팀 0.97110 파이프라인의 추론 단계 기법을 더합니다.

| 단계 | 내용 | 비교 |
| --- | --- | --- |
| E8-1 **Order4** | 선지 순서 `abcd·bcda·cdab·dabc`로 4번 풀고, 확률을 원래 선지로 되돌려 평균 | E7-C 단독 |
| E8-2 **Gate → Zoom → Blend** | 최댓값 < 0.85 · 1·2위 차이 < 0.25 · 순서 불일치 중 하나면 고해상도로 다시 Order4, 기존과 50%씩 섞음 | E8-1 |
| E8-3 **제출** | 같은 파이프라인으로 test 6,714 | LB 0.97140(E8-A) · 0.97110(팀) |

**기준값을 고치지 마세요.** Gate·Blend 값은 팀이 자기 검증 1,343으로 정한 값입니다.
holdout 결과를 보고 바꾸면 holdout이 더는 독립 확인이 아니게 됩니다.

| 단계 | 시간 (추정) |
| --- | ---: |
| 설치·데이터·원본 다운로드 | 15~20분 |
| 병합 (Drive 어댑터 → bf16 원본) | 약 5분 |
| holdout Order4 (1,300 × 4) | 약 15분 |
| holdout Zoom (약 100 × 4) | 약 3분 |
| test Order4 + Zoom (6,714 × 4 + 약 500 × 4) | 약 1시간 40분 |

추론은 순서마다 256문항 묶음으로 Drive에 저장되므로, 끊겨도 다시 실행하면 이어 갑니다.

## 준비물

- A100 80GB 런타임
- Drive `MyDrive/SSAFY/AI_Challenge2/ssafy-16-2-ai.zip`
- Drive `MyDrive/SSAFY/AI_Challenge2/output/exp_E7_27b_C/adapter/` (E7-C 학습 결과)
- 비교용: `output/exp_E7_27b_C/holdout/`, `output/exp_E6_zeroshot_train/` (없으면 해당 비교만 건너뜀)

## 1. 설치

새 런타임에서 가장 먼저 실행하고, 끝나면 **런타임 → 세션 다시 시작** 후 2번 셀부터 실행하세요.

In [ ]:
!pip -q install -U vllm
!pip -q install -U peft "huggingface_hub[hf_xet]" pillow pandas
# E7과 같은 충돌 회피: torchaudio(CUDA 불일치로 transformers import 실패), torchao 0.10(peft와 충돌)
!pip -q uninstall -y torchaudio torchao

In [ ]:
import gc, json, os, sys, time
from pathlib import Path

import pandas as pd
import torch
import transformers
import peft
import vllm

print("vllm", vllm.__version__, "/ transformers", transformers.__version__, "/ peft", peft.__version__)
for mod in ("torchaudio", "torchao"):
    try:
        __import__(mod)
        raise RuntimeError(f"{mod}가 남아 있습니다. `!pip uninstall -y {mod}` 후 세션을 다시 시작하세요")
    except ImportError:
        pass
assert torch.cuda.is_available(), "GPU 런타임을 선택하세요"
GPU_GIB = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {torch.cuda.get_device_name(0)} / {GPU_GIB:.1f} GiB")
assert GPU_GIB >= 70, "27B bf16 병합·추론은 A100 80GB가 필요합니다"

## 2. Drive · 데이터 · 공통 코드

In [ ]:
from google.colab import drive
print("[1/3] Drive 연결 중… (권한 창이 뜨면 허용)", flush=True)
drive.mount("/content/drive")

DRIVE_BASE = Path("/content/drive/MyDrive/SSAFY/AI_Challenge2")
ZIP_PATH = DRIVE_BASE / "ssafy-16-2-ai.zip"
ZIP_PASSWORD = "ocrisnotenough"
assert ZIP_PATH.is_file(), f"zip을 찾지 못했습니다: {ZIP_PATH}"


def find_data_root():
    for parent, dirs, files in os.walk("/content"):
        dirs[:] = [d for d in dirs if d not in {"drive", ".cache", "sample_data", "models", "merged"}]
        if "test.csv" in files and "train.csv" in files:
            return Path(parent)
    return None


if find_data_root() is None:
    import subprocess
    print("[2/3] 데이터 zip(1.8 GB) 압축 해제 중… 출력 없이 1~5분 걸립니다. "
          "왼쪽 파일 탭 /content에 train/ 폴더가 생기면 진행 중입니다.", flush=True)
    t0 = time.time()
    subprocess.run(["7z", "x", "-y", f"-p{ZIP_PASSWORD}", "-o/content/", str(ZIP_PATH)],
                   check=True, stdout=subprocess.DEVNULL)
    print(f"      압축 해제 {time.time() - t0:.0f}초", flush=True)
else:
    print("[2/3] 데이터가 이미 풀려 있어 건너뜁니다", flush=True)
DATA_ROOT = find_data_root()
assert DATA_ROOT is not None, "train/test.csv를 찾지 못했습니다"
print("[3/3] 공통 코드 준비·자체 테스트", flush=True)

SOURCES = {'common_eval.py': '"""실험 공통 코드.\n\n모든 실험(A: OCR+LLM, B: VLM 단독, C: OCR+VLM, D: LLM blind)이\n동일한 검증 분할 / 질문 프롬프트 / 출력 파서 / 분석 지표를 쓰도록 한곳에 모읍니다.\n각 노트북이 코드를 복사하면 조용히 어긋나므로 반드시 import 해서 사용하세요.\n"""\nimport hashlib\nimport json\nimport re\nfrom pathlib import Path\n\nimport pandas as pd\n\nDATA_DIR = Path("data")\nCHOICES = list("abcd")\nSYSTEM = ("You are a visual question answering assistant. "\n          "Answer with exactly one lowercase letter: a, b, c, or d. No explanation.")\n# 실험 A가 만든 기준 분할. 모든 실험은 이 val 672문항을 그대로 씁니다.\nBASELINE_RUN = Path("output/paddleocr_vl16_halftrain/full")\n\n\nANSWER_INSTRUCTION = "이미지를 보고 정답 선지의 영문 소문자 한 글자만 답하세요."\n# CoT에서는 "한 글자만" 지시와 "단계적으로 설명" 지시가 충돌하므로 마지막 줄을 통째로 교체합니다.\nCOT_INSTRUCTION = ("이미지에서 근거가 되는 부분을 찾아 간단히 단계적으로 설명한 뒤, "\n                   "마지막 줄에 반드시 `정답: x` 형식으로 a, b, c, d 중 한 글자만 쓰세요.")\n\n\n# E6: 정답은 사진 속 사실이 아니라 여러 교육생이 일치해서 고른 답입니다(응답이 갈린 문항은 제외됨).\n# E1 오답 중 \'라벨 오류\'로 본 문항도 사람 기준으로는 일반인 상식(그리스 문자 -> 라틴어)이나\n# 눈에 띄는 쪽(두 수건 중 앞의 흰 수건)을 고른 것이었습니다. 그 기준을 지시문으로 알려 봅니다.\nCONSENSUS_INSTRUCTION = ("이 문제의 정답은 여러 사람이 같은 사진을 보고 일치해서 고른 답입니다. "\n                         "사진에 적힌 표기를 그대로 따르고, 전문 지식보다 일반적인 상식과 "\n                         "눈에 잘 띄는 정보를 기준으로 사람들이 가장 많이 고를 선지의 "\n                         "영문 소문자 한 글자만 답하세요.")\n\n\ndef make_prompt(row, cot=False, consensus=False):\n    """질문/선지 렌더링. 본문은 모든 실험에서 동일하고 마지막 지시문만 달라집니다."""\n    assert not (cot and consensus), "CoT와 합의 지시문은 함께 쓰지 않습니다"\n    instruction = COT_INSTRUCTION if cot else CONSENSUS_INSTRUCTION if consensus else ANSWER_INSTRUCTION\n    return (f"질문: {row[\'question\']}\\n"\n            f"a. {row[\'a\']}\\nb. {row[\'b\']}\\nc. {row[\'c\']}\\nd. {row[\'d\']}\\n"\n            + instruction)\n\n\ndef parse_choice(raw):\n    """출력이 정확히 선지 한 글자일 때만 인정하고, 나머지는 INVALID로 남깁니다."""\n    raw = re.sub(r"<think>.*?</think>", "", str(raw), flags=re.S).strip()\n    raw = re.sub(r"^(?:정답|답|answer)\\s*[:：]\\s*", "", raw, flags=re.I).strip()\n    m = re.fullmatch(r"(?:\\(?([abcd])\\)?[.。]?)", raw, flags=re.I)\n    return m.group(1).lower() if m else "INVALID"\n\n\n# CoT 모드에서는 system 프롬프트의 \'설명 금지\'를 풀어야 합니다.\nSYSTEM_COT = ("You are a visual question answering assistant. "\n              "Think briefly, then end your reply with a final line "\n              "in the form `정답: x` where x is one of a, b, c, d.")\n\n_FINAL_RE = re.compile(r"(?:정답|답|answer|final answer)\\s*[:：]\\s*\\(?\\s*([abcd])\\b",\n                       re.IGNORECASE)\n_BARE_RE = re.compile(r"(?<![a-z])\\(?([abcd])\\)?(?![a-z])", re.IGNORECASE)\n\n\ndef parse_choice_cot(raw):\n    """CoT 출력에서 최종 선지를 추출합니다.\n\n    parse_choice()는 출력이 정확히 한 글자일 때만 인정하므로 CoT 문장은 전부 INVALID가 됩니다.\n    여기서는 `정답: x` 형태를 우선 찾고, 없으면 마지막 줄의 단독 a~d를 씁니다.\n    추론 과정에 등장하는 선지 언급을 최종 답으로 오인하지 않도록 뒤에서부터 찾습니다.\n    """\n    text = re.sub(r"<think>.*?</think>", " ", str(raw), flags=re.S)\n    matches = _FINAL_RE.findall(text)\n    if matches:\n        return matches[-1].lower()\n    # `정답:` 표기가 없으면 마지막 비어 있지 않은 줄에서만 단독 글자를 찾습니다.\n    for line in reversed([l.strip() for l in text.splitlines() if l.strip()]):\n        found = _BARE_RE.findall(line)\n        if len(found) == 1:\n            return found[0].lower()\n        if found:\n            break            # 한 줄에 여러 선지가 있으면 최종 답으로 볼 수 없습니다.\n    return "INVALID"\n\n\n# --- E3: 선지를 감추고 먼저 읽게 하는 2단계 프롬프트 -------------------------\n# E1/E2는 질문과 선지 4개를 한꺼번에 줍니다. 그러면 선지가 판독을 유도합니다.\n# 실제로 E1 오답 중 CoT로도 못 고친 22건에서 정답 문자열이 추론문에 등장한 것은 5건뿐이고,\n# 모델은 간판의 \'미술랭투스타\'를 흔한 표기 \'미슐랭 스타\'로 교정해 읽었습니다.\n# 1단계에서 선지를 감추면 그 닻이 사라지는지 보는 것이 E3의 가설입니다.\n\nSYSTEM_READ = ("You are a careful transcription assistant. "\n               "Write down exactly what is printed in the image. "\n               "Never correct spelling, never normalize to a more common wording, never guess.")\n\nREAD_INSTRUCTION = ("위 질문에 답하는 데 필요한 부분의 글자를 이미지에 적힌 그대로 옮겨 적으세요. "\n                    "맞춤법을 고치거나 더 흔한 표현으로 바꾸지 마세요. "\n                    "글자가 보이지 않으면 \'확인 불가\'라고만 쓰세요.")\n\nMATCH_INSTRUCTION = "전사 내용과 이미지를 함께 보고 정답 선지의 영문 소문자 한 글자만 답하세요."\n\n\ndef make_read_prompt(row):\n    """1단계. 선지를 넣지 않습니다. 넣는 순간 실험이 무의미해집니다."""\n    return f"질문: {row[\'question\']}\\n" + READ_INSTRUCTION\n\n\ndef make_match_prompt(row, transcript):\n    """2단계. 1단계 전사를 앞에 붙이고 나머지는 E1과 동일하게 렌더링합니다."""\n    return (f"이미지에서 읽은 글자: {str(transcript).strip()}\\n\\n"\n            f"질문: {row[\'question\']}\\n"\n            f"a. {row[\'a\']}\\nb. {row[\'b\']}\\nc. {row[\'c\']}\\nd. {row[\'d\']}\\n"\n            + MATCH_INSTRUCTION)\n\n\nIMAGE_MAX_EDGE = 960  # 원본 노트북의 공통 이미지 규칙. B와 C가 반드시 같은 값을 써야 비교가 성립합니다.\n\n\ndef open_image(row, max_edge=IMAGE_MAX_EDGE):\n    """원본 노트북 open_image와 동일: RGB 변환 후 긴 변을 max_edge로 제한."""\n    from PIL import Image\n    with Image.open(DATA_DIR / str(row["path"])) as im:\n        im = im.convert("RGB")\n        if max_edge is not None and max(im.size) > max_edge:\n            scale = max_edge / max(im.size)\n            im = im.resize((max(1, round(im.width * scale)), max(1, round(im.height * scale))),\n                           Image.Resampling.LANCZOS)\n        return im\n\n\ndef ids_sha256(ids):\n    return hashlib.sha256(json.dumps(list(ids)).encode()).hexdigest()\n\n\ndef load_val(run_dir=BASELINE_RUN):\n    """기준 실험이 저장한 검증 분할을 그대로 읽고, ID 해시로 동일 분할임을 확인합니다.\n\n    sklearn 버전이 달라져도 재계산하지 않으므로 분할이 어긋날 수 없습니다.\n    """\n    run_dir = Path(run_dir)\n    val_path = run_dir / "train_half_val.csv"\n    if not val_path.is_file():\n        raise FileNotFoundError(\n            f"기준 분할이 없습니다: {val_path}. 먼저 실험 A 노트북의 분할 셀을 실행하세요.")\n    val_df = pd.read_csv(val_path, keep_default_na=False)\n    config_path = run_dir / "run_config.json"\n    if config_path.is_file():\n        expected = json.loads(config_path.read_text(encoding="utf-8")).get("validation_ids_sha256")\n        actual = ids_sha256(val_df.id.tolist())\n        if expected and expected != actual:\n            raise ValueError(f"검증 분할이 기준과 다릅니다: {actual} != {expected}")\n        print(f"검증 분할 해시 일치: {actual[:16]}…")\n    missing = [p for p in val_df.path if not (DATA_DIR / p).is_file()]\n    if missing:\n        raise FileNotFoundError(f"이미지 누락 {len(missing)}개. 예: {missing[:3]}")\n    print(f"검증 문항 {len(val_df)}개 / 정답 분포:\\n{val_df.answer.value_counts().sort_index()}")\n    return val_df\n\n\ndef load_ocr_evidence(run_dir=BASELINE_RUN, mode="ocr_llm"):\n    """실험 A가 남긴 stage1.jsonl(Spotting 결과)을 id -> 텍스트로 읽습니다. 실험 C가 사용."""\n    path = Path(run_dir) / mode / "stage1.jsonl"\n    if not path.is_file():\n        raise FileNotFoundError(f"OCR 결과가 없습니다: {path}. 실험 A를 먼저 완료하세요.")\n    evidence = {}\n    with path.open(encoding="utf-8") as f:\n        for line in f:\n            item = json.loads(line)\n            evidence[item["id"]] = item["stage1_raw"]\n    empty = sum(1 for v in evidence.values() if not v.strip())\n    print(f"OCR 근거 {len(evidence)}건 로드 (빈 결과 {empty}건)")\n    return evidence\n\n\ndef save_analysis(pred_df, name, out_dir, extra=None):\n    """정확도/macro-F1/혼동행렬/분류리포트를 저장하고 지표 dict를 반환합니다."""\n    from sklearn.metrics import (accuracy_score, classification_report,\n                                 confusion_matrix, f1_score)\n    import matplotlib.pyplot as plt\n\n    dest = Path(out_dir)\n    dest.mkdir(parents=True, exist_ok=True)\n    pred_df.to_csv(dest / "validation_predictions.csv", index=False, encoding="utf-8-sig")\n    pred_df.loc[~pred_df["correct"]].to_csv(dest / "validation_errors.csv",\n                                            index=False, encoding="utf-8-sig")\n    # INVALID를 열로 유지하여 오답이 사라지거나 임의로 a가 되지 않도록 합니다.\n    labels = CHOICES + ["INVALID"]\n    cm = confusion_matrix(pred_df["answer"], pred_df["pred"], labels=labels)\n    cm_df = pd.DataFrame(cm, index=[f"true_{k}" for k in labels],\n                         columns=[f"pred_{k}" for k in labels])\n    cm_df.to_csv(dest / "confusion_matrix.csv", encoding="utf-8-sig")\n    class_df = pd.DataFrame(classification_report(\n        pred_df["answer"], pred_df["pred"], labels=CHOICES,\n        output_dict=True, zero_division=0)).T\n    class_df.to_csv(dest / "classification_report.csv", encoding="utf-8-sig")\n    score = {\n        "experiment": name,\n        "validation_rows": len(pred_df),\n        "correct": int(pred_df.correct.sum()),\n        "accuracy": float(accuracy_score(pred_df.answer, pred_df.pred)),\n        "macro_f1": float(f1_score(pred_df.answer, pred_df.pred, labels=CHOICES,\n                                   average="macro", zero_division=0)),\n        "invalid": int((pred_df.pred == "INVALID").sum()),\n        "invalid_rate": float((pred_df.pred == "INVALID").mean()),\n        "mean_inference_seconds": float(pred_df.seconds.mean()),\n        "total_inference_seconds": float(pred_df.seconds.sum()),\n        "validation_ids_sha256": ids_sha256(pred_df.id.tolist()),\n    }\n    score.update(extra or {})\n    (dest / "metrics.json").write_text(json.dumps(score, ensure_ascii=False, indent=2),\n                                       encoding="utf-8")\n    print(f"[{name}] Accuracy={score[\'accuracy\']:.4f} / macro-F1={score[\'macro_f1\']:.4f} "\n          f"/ INVALID={score[\'invalid\']}")\n    print(class_df.loc[CHOICES, ["precision", "recall", "f1-score", "support"]].round(3))\n    fig, ax = plt.subplots(figsize=(7, 5))\n    ax.imshow(cm[:4, :], cmap="Blues")\n    ax.set_xticks(range(5), labels)\n    ax.set_yticks(range(4), CHOICES)\n    ax.set_xlabel("Predicted")\n    ax.set_ylabel("True")\n    ax.set_title(f"{name} | Accuracy={score[\'accuracy\']:.3f}")\n    for i in range(4):\n        for j in range(5):\n            ax.text(j, i, str(cm[i, j]), ha="center", va="center")\n    fig.tight_layout()\n    fig.savefig(dest / "confusion_matrix.png", dpi=180)\n    plt.show()\n    plt.close(fig)\n    return score\n\n\ndef _self_test():\n    assert parse_choice("c") == "c"\n    assert parse_choice("a 또는 b") == "INVALID"\n    assert parse_choice("") == "INVALID"\n    assert parse_choice("정답: D") == "d"\n    assert parse_choice("간판을 보면 ... 정답: c") == "INVALID"   # 엄격 파서는 문장을 거부\n    # CoT 파서\n    assert parse_choice_cot("a 와 b 를 비교하면...\\n정답: c") == "c"\n    assert parse_choice_cot("생각: b 인 것 같지만 아니다.\\n정답: (D)") == "d"\n    assert parse_choice_cot("설명이 길다\\n\\nb") == "b"\n    assert parse_choice_cot("a 또는 b 중 하나") == "INVALID"\n    assert parse_choice_cot("") == "INVALID"\n    assert parse_choice_cot("정답: a 이지만 정답: b 로 정정") == "b"   # 마지막 선언을 채택\n    row = {"question": "Q", "a": "A", "b": "B", "c": "C", "d": "D"}\n    assert make_prompt(row).endswith(ANSWER_INSTRUCTION)\n    assert make_prompt(row, cot=True).endswith(COT_INSTRUCTION)\n    # 질문/선지 본문은 두 모드가 완전히 동일해야 비교가 성립합니다.\n    assert make_prompt(row).rsplit(chr(10), 1)[0] == make_prompt(row, cot=True).rsplit(chr(10), 1)[0]\n    assert make_prompt(row, consensus=True).endswith(CONSENSUS_INSTRUCTION)\n    assert make_prompt(row).rsplit(chr(10), 1)[0] == make_prompt(row, consensus=True).rsplit(chr(10), 1)[0]\n    # E3 1단계는 선지가 새면 안 됩니다.\n    read = make_read_prompt(row)\n    assert read.endswith(READ_INSTRUCTION)\n    for key in CHOICES:\n        assert f"{key}. {row[key]}" not in read, "1단계 프롬프트에 선지가 들어갔습니다"\n    # E3 2단계는 전사문만 앞에 붙고 질문/선지 본문은 E1과 동일해야 합니다.\n    match = make_match_prompt(row, "테스트 전사")\n    assert "테스트 전사" in match and match.endswith(MATCH_INSTRUCTION)\n    body = lambda s: s.split("질문: ", 1)[1].rsplit(chr(10), 1)[0]\n    assert body(match) == body(make_prompt(row)), "2단계 질문/선지 본문이 E1과 다릅니다"\n    print("공통 프롬프트/파서 OK (엄격 + CoT + E3 2단계)")\n\n\nif __name__ == "__main__":\n    _self_test()\n', 'vlm_runtime.py': '"""E4 이후 실험이 공유하는 Qwen3.5 VLM 적재·추론·짝 비교 코드.\n\nE1~E3는 이 코드를 노트북마다 복사했습니다. E4(27B 제로샷)와 E5(QLoRA)는 E1과\n문항 단위로 맞대야 하므로 적재·메시지·생성 규칙을 여기 한 벌만 둡니다.\n프롬프트와 파서는 여전히 common_eval에서 옵니다.\n"""\nimport json\nimport math\nimport time\nfrom math import comb\nfrom pathlib import Path\n\nimport pandas as pd\nimport torch\n\nimport common_eval as ce\n\nE1_PRED = Path("output/exp_E1_qwen35_9b/predictions.jsonl")\nE1_TAXONOMY = Path("output/exp_E1_qwen35_9b/error_taxonomy.csv")\n\n\ndef check_weights(model_dir):\n    """config와 index에 적힌 shard가 전부 있는지 확인합니다. 다운로드 중단을 잡습니다."""\n    model_dir = Path(model_dir)\n    assert (model_dir / "config.json").is_file(), f"로컬 모델을 준비하세요: {model_dir}"\n    index = model_dir / "model.safetensors.index.json"\n    if index.is_file():\n        need = sorted(set(json.loads(index.read_text())["weight_map"].values()))\n        miss = [f for f in need if not (model_dir / f).is_file()]\n        assert not miss, f"가중치 shard {len(miss)}/{len(need)}개 누락: {miss[:2]}"\n    assert list(model_dir.glob("*.safetensors")), f"가중치(.safetensors)가 없습니다: {model_dir}"\n\n\ndef build_device_map(model_dir, cpu_modules=(), cpu_last_layers=0):\n    """GPU에 다 못 올릴 때 CPU로 보낼 모듈을 명시합니다.\n\n    device_map="auto"는 앞에서부터 채우므로 비전 인코더나 앞쪽 층이 CPU로 밀릴 수 있습니다.\n    27B는 임베딩과 lm_head(각 약 2.5 GiB, 양자화 대상 아님)만 빼면 4bit 본체가 16 GiB에 들어가므로\n    그 둘을 먼저 내리고, 그래도 부족할 때만 마지막 층부터 내립니다.\n    """\n    cfg = json.loads((Path(model_dir) / "config.json").read_text(encoding="utf-8"))\n    n_layers = cfg["text_config"]["num_hidden_layers"]\n    lm = "model.language_model"\n    dmap = {"model.visual": 0, f"{lm}.embed_tokens": 0, f"{lm}.norm": 0,\n            f"{lm}.rotary_emb": 0, "lm_head": 0}\n    for i in range(n_layers):\n        dmap[f"{lm}.layers.{i}"] = "cpu" if i >= n_layers - cpu_last_layers else 0\n    for name in cpu_modules:\n        assert name in dmap, f"알 수 없는 모듈: {name}"\n        dmap[name] = "cpu"\n    return dmap\n\n\ndef execute_on_cpu(module, out_device="cuda:0"):\n    """accelerate 오프로딩 훅을 떼고 모듈을 CPU에서 직접 계산하게 바꿉니다.\n\n    기본 오프로딩은 forward마다 가중치를 GPU로 복사합니다. 27B의 임베딩·lm_head는 합쳐 5 GiB라\n    문항당 수 초가 복사에 들어갑니다. 임베딩은 조회뿐이고 lm_head는 생성 중 마지막 토큰 하나만\n    계산하므로 CPU에서 돌리고 결과만 GPU로 보내는 편이 훨씬 빠릅니다.\n    """\n    from accelerate.hooks import remove_hook_from_module\n    from accelerate.utils import set_module_tensor_to_device\n\n    hook = getattr(module, "_hf_hook", None)\n    weights = getattr(hook, "weights_map", None)\n    names = [n for n, _ in module.named_parameters()]\n    values = {n: weights[n] for n in names} if weights is not None else {}\n    remove_hook_from_module(module, recurse=True)\n    for n in names:\n        if n in values:\n            set_module_tensor_to_device(module, n, "cpu", value=values[n])\n    assert all(p.device.type == "cpu" for p in module.parameters()), "CPU 적재 실패"\n    module.register_forward_pre_hook(\n        lambda m, args, kwargs: (tuple(a.to("cpu") if torch.is_tensor(a) else a for a in args),\n                                 {k: v.to("cpu") if torch.is_tensor(v) else v\n                                  for k, v in kwargs.items()}), with_kwargs=True)\n    module.register_forward_hook(lambda m, args, out: out.to(out_device))\n\n\ndef load_vlm(model_dir, four_bit=True, dtype=torch.bfloat16, cpu_modules=(), cpu_last_layers=0,\n             adapter_dir=None):\n    """E1과 같은 nf4 설정으로 적재합니다. adapter_dir를 주면 LoRA를 얹습니다."""\n    from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3_5ForConditionalGeneration\n\n    check_weights(model_dir)\n    proc = AutoProcessor.from_pretrained(str(model_dir), local_files_only=True)\n    offload = bool(cpu_modules) or cpu_last_layers > 0\n    dmap = (build_device_map(model_dir, cpu_modules, cpu_last_layers) if offload else {"": 0})\n    kwargs = dict(local_files_only=True, dtype=dtype, device_map=dmap)\n    if four_bit:\n        kwargs["quantization_config"] = BitsAndBytesConfig(\n            load_in_4bit=True, bnb_4bit_quant_type="nf4",\n            bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=dtype,\n            # CPU로 보낸 모듈은 양자화하지 않고 dtype 그대로 둡니다.\n            llm_int8_enable_fp32_cpu_offload=offload)\n    model = Qwen3_5ForConditionalGeneration.from_pretrained(str(model_dir), **kwargs)\n    for name in cpu_modules:\n        execute_on_cpu(model.get_submodule(name))\n    if adapter_dir is not None:\n        from peft import PeftModel\n        model = PeftModel.from_pretrained(model, str(adapter_dir), is_trainable=False)\n    return model.eval(), proc\n\n\ndef build_messages(row, image, consensus=False):\n    """E1과 글자 단위로 같은 메시지. 학습(E5·E7)도 이 함수로 입력을 만들어야 분포가 맞습니다.\n\n    consensus=True면 마지막 지시문만 E6의 합의 지시문으로 바뀝니다.\n    """\n    return [{"role": "system", "content": [{"type": "text", "text": ce.SYSTEM}]},\n            {"role": "user", "content": [{"type": "image", "image": image},\n                                         {"type": "text",\n                                          "text": ce.make_prompt(row, consensus=consensus)}]}]\n\n\ndef encode(proc, messages, enable_thinking=False):\n    return proc.apply_chat_template(messages, add_generation_prompt=True, tokenize=True,\n                                    return_dict=True, return_tensors="pt",\n                                    enable_thinking=enable_thinking)\n\n\ndef letter_token_ids(proc):\n    """a~d 각각이 첫 토큰이 될 수 있는 id 후보. 대소문자·앞 공백 변형을 모읍니다."""\n    tok = proc.tokenizer\n    table = {}\n    for ch in ce.CHOICES:\n        cand = set()\n        for form in (ch, " " + ch, ch.upper(), " " + ch.upper()):\n            enc = tok.encode(form, add_special_tokens=False)\n            if enc:\n                cand.add(int(enc[0]))\n        assert cand, f"\'{ch}\'의 토큰 id를 찾지 못했습니다"\n        table[ch] = sorted(cand)\n    return table\n\n\n@torch.inference_mode()\ndef answer(model, proc, inputs, limit, letters, device="cuda:0"):\n    """E1과 같은 탐욕 생성에 더해, 첫 토큰의 a~d 로그확률을 함께 돌려줍니다.\n\n    예측은 생성 결과를 parse_choice로 읽은 값 그대로이고(E1과 동일), 로그확률은\n    나중에 모델 앙상블·선지 순서 TTA에 쓰려고 추가 비용 없이 남기는 기록입니다.\n    """\n    inputs = inputs.to(device)\n    prefix_len = inputs["input_ids"].shape[-1]\n    out = model.generate(**inputs, max_new_tokens=limit, do_sample=False, use_cache=True,\n                         output_logits=True, return_dict_in_generate=True)\n    new = out.sequences[0, prefix_len:]\n    eos = model.generation_config.eos_token_id\n    eos_ids = [] if eos is None else ([eos] if isinstance(eos, int) else list(eos))\n    truncated = len(new) >= limit and (not len(new) or int(new[-1]) not in eos_ids)\n    logp = torch.log_softmax(out.logits[0][0].float(), dim=-1)\n    letter_logp = {ch: max(float(logp[i]) for i in ids) for ch, ids in letters.items()}\n    raw = proc.decode(new, skip_special_tokens=True).strip()\n    return raw, bool(truncated), int(len(new)), letter_logp\n\n\ndef run_predictions(model, proc, rows, pred_path, limit=24, resume=True, desc="predict"):\n    """문항마다 predictions.jsonl에 기록하고 flush합니다. 다시 부르면 끝난 문항을 건너뜁니다."""\n    from tqdm.auto import tqdm\n\n    pred_path = Path(pred_path)\n    done = set()\n    if resume and pred_path.is_file():\n        with pred_path.open(encoding="utf-8") as f:\n            done = {json.loads(line)["id"] for line in f if line.strip()}\n        print(f"이어서 진행: 이미 끝난 {len(done)}개 건너뜀")\n    elif not resume and pred_path.is_file():\n        pred_path.unlink()\n    todo = [r for r in rows if r["id"] not in done]\n    print(f"남은 {len(todo)}개")\n\n    letters = letter_token_ids(proc)\n    torch.cuda.reset_peak_memory_stats()\n    with pred_path.open("a", encoding="utf-8") as log:\n        for row in tqdm(todo, desc=desc):\n            torch.cuda.synchronize(); start = time.perf_counter()\n            im = ce.open_image(row)\n            inputs = encode(proc, build_messages(row, im))\n            raw, capped, n_tok, letter_logp = answer(model, proc, inputs, limit, letters)\n            torch.cuda.synchronize(); elapsed = time.perf_counter() - start\n            log.write(json.dumps(dict(\n                id=row["id"], raw_output=raw, pred=ce.parse_choice(raw), letter_logprobs=letter_logp,\n                answer_truncated=capped, answer_tokens=n_tok, seconds=elapsed,\n                image_width=im.width, image_height=im.height), ensure_ascii=False) + "\\n")\n            log.flush()\n            del inputs, im\n    return torch.cuda.max_memory_allocated() / 1024**3\n\n\ndef read_predictions(pred_path, val_df):\n    res = pd.read_json(pred_path, lines=True).drop_duplicates("id", keep="last")\n    pred_df = val_df.merge(res, on="id", validate="one_to_one", sort=False)\n    pred_df["correct"] = pred_df.pred == pred_df.answer\n    return pred_df\n\n\ndef mcnemar_exact(b, c):\n    n = b + c\n    if n == 0:\n        return 1.0\n    return min(1.0, 2 * sum(comb(n, k) for k in range(min(b, c) + 1)) / 2 ** n)\n\n\ndef paired_vs_e1(pred_df, name, e1_pred=E1_PRED, taxonomy=E1_TAXONOMY):\n    """E1과 문항 단위로 맞댑니다. 정확도 두 개를 나란히 놓는 것만으로는 부족합니다.\n\n    E1 오답 분류(error_taxonomy.csv)가 있으면 분류별로 몇 개를 고쳤는지도 봅니다.\n    27B는 판독·위치 관계를, QLoRA는 \'적힌 그대로\' 유형을 고칠 것이라는 예상을 여기서 확인합니다.\n    """\n    if not Path(e1_pred).is_file():\n        print("E1 예측이 없어 짝 비교를 건너뜁니다:", e1_pred)\n        return None\n    e1 = pd.read_json(e1_pred, lines=True).drop_duplicates("id", keep="last")\n    joined = pred_df.merge(e1[["id", "pred"]].rename(columns={"pred": "pred_e1"}), on="id")\n    ok1 = joined.pred_e1 == joined.answer\n    ok2 = joined.pred == joined.answer\n    n = len(joined)\n    b = int((ok1 & ~ok2).sum())\n    c = int((~ok1 & ok2).sum())\n    diff = (c - b) / n\n    se = math.sqrt(b + c) / n\n    p = mcnemar_exact(b, c)\n    print(f"[{n}문항]")\n    print(f"  E1   {ok1.mean():.4f} ({int(ok1.sum())}/{n})")\n    print(f"  {name} {ok2.mean():.4f} ({int(ok2.sum())}/{n})")\n    print(f"  E1만 정답 {b} / {name}만 정답 {c} / 둘 다 오답 {int((~ok1 & ~ok2).sum())}")\n    print(f"  {name} - E1 = {diff * 100:+.2f}pp  95%CI "\n          f"[{(diff - 1.96 * se) * 100:+.2f}pp, {(diff + 1.96 * se) * 100:+.2f}pp]")\n    print(f"  McNemar exact p = {p:.4g} -> "\n          + ("차이가 유의합니다" if p < 0.05 else "두 조건은 구분되지 않습니다"))\n\n    if Path(taxonomy).is_file():\n        tax = pd.read_csv(taxonomy, encoding="utf-8")\n        t = tax.merge(joined[["id", "pred", "answer"]], on="id", how="left")\n        t["fixed"] = t.pred == t.answer\n        table = t.groupby("category", sort=False).agg(E1_errors=("id", "size"),\n                                                      fixed=("fixed", "sum"))\n        print("\\nE1 오답 분류별로 고친 수 (label_error·ambiguous는 고쳐도 운에 가깝습니다)")\n        print(table.to_string())\n    return dict(n=n, e1_only=b, new_only=c, diff=diff, p=p)\n\n\ndef _self_test():\n    assert mcnemar_exact(0, 0) == 1.0\n    assert mcnemar_exact(11, 11) == 1.0\n    assert abs(mcnemar_exact(11, 17) - 0.3449) < 1e-3   # E2 vs E1 기록값\n    row = {"question": "Q", "a": "A", "b": "B", "c": "C", "d": "D"}\n    assert build_messages(row, None)[1]["content"][1]["text"] == ce.make_prompt(row)\n    print("vlm_runtime OK")\n\n\nif __name__ == "__main__":\n    _self_test()\n', 'qlora.py': '"""E5 QLoRA 학습 코드.\n\n핵심 규칙은 하나입니다. **학습 입력은 추론 입력과 토큰 단위로 같아야 합니다.**\n그래서 프롬프트는 vlm_runtime.build_messages + encode(add_generation_prompt=True)로 만들고,\n그 뒤에 정답 글자와 <|im_end|>만 붙입니다. 채팅 템플릿에 assistant 메시지를 넣어 렌더링하면\nthinking 블록 처리가 추론 때와 달라질 수 있어 쓰지 않습니다.\n"""\nimport json\nimport math\nfrom pathlib import Path\n\nimport pandas as pd\nimport torch\n\nimport common_eval as ce\nimport vlm_runtime as vr\n\n# 언어 모델 쪽 선형층만. 비전 인코더와 merger는 얼립니다(E1 오답의 다수가 \'읽고 나서 고치는\' 쪽이라\n# 우선 언어 쪽 습관을 바꿔 보고, 판독 실패가 안 줄면 그때 merger를 엽니다).\nLORA_TARGET = (r"model\\.language_model\\.layers\\.\\d+\\."\n               r"(self_attn\\.(q_proj|k_proj|v_proj|o_proj)"\n               r"|linear_attn\\.(in_proj_qkv|in_proj_z|out_proj)"\n               r"|mlp\\.(gate_proj|up_proj|down_proj))")\n\n\ndef load_train_rows(val_df, train_csv=ce.DATA_DIR / "train.csv", seed=42):\n    """train.csv에서 검증 672문항을 뺀 나머지 전부. 검증과 ID·이미지가 겹치지 않는지 확인합니다."""\n    train = pd.read_csv(train_csv, keep_default_na=False)\n    val_ids, val_paths = set(val_df.id), set(val_df.path)\n    fit = train[~train.id.isin(val_ids)].copy()\n    assert not set(fit.path) & val_paths, "학습 이미지가 검증과 겹칩니다"\n    assert len(fit) + len(val_ids) == len(train), "검증 ID 일부가 train.csv에 없습니다"\n    assert set(fit.answer) <= set(ce.CHOICES), "정답 형식이 a~d가 아닙니다"\n    return fit.sample(frac=1.0, random_state=seed).reset_index(drop=True)\n\n\nDEV_VOTES = [f"answer{i}" for i in range(1, 6)]\n\n\ndef dev_majority(row, min_votes=3):\n    """dev의 5명 응답 중 min_votes표 이상 받은 단독 최다 답. 없으면 None."""\n    votes = [row[c] for c in DEV_VOTES if row.get(c) in ce.CHOICES]\n    if not votes:\n        return None\n    counts = {ch: votes.count(ch) for ch in set(votes)}\n    best = max(counts.values())\n    leaders = [ch for ch, k in counts.items() if k == best]\n    return leaders[0] if best >= min_votes and len(leaders) == 1 else None\n\n\ndef rows_from_manifest(manifest, train_df, dev_df, holdout_ids, min_votes=3):\n    """학습 구성 목록(id, source)을 학습 행으로 바꿉니다. 정답 글자는 저장소에 두지 않고 여기서 붙입니다.\n\n    train 문항은 train.csv의 정답을, dev 문항은 다수결(min_votes표 이상, 단독 최다)을 씁니다.\n    holdout 문항이 섞이면 평가가 무의미해지므로 멈춥니다.\n    """\n    assert set(manifest.source) <= {"train", "dev"}, f"알 수 없는 source: {set(manifest.source)}"\n    leaked = set(manifest.id) & set(holdout_ids)\n    assert not leaked, f"holdout 문항 {len(leaked)}개가 학습 구성에 들어 있습니다: {sorted(leaked)[:3]}"\n    tr = train_df.set_index("id")\n    dv = dev_df.set_index("id")\n    rows, dropped = [], 0\n    for item in manifest.itertuples():\n        if item.source == "train":\n            rows.append({"id": item.id, **tr.loc[item.id].to_dict()})\n        else:\n            r = dv.loc[item.id].to_dict()\n            answer = dev_majority(r, min_votes)\n            if answer is None:\n                dropped += 1\n                continue\n            rows.append({"id": item.id, **{k: v for k, v in r.items() if k not in DEV_VOTES},\n                         "answer": answer})\n    if dropped:\n        print(f"dev 다수결이 {min_votes}표에 못 미치거나 동률이라 뺀 문항 {dropped}개")\n    return rows\n\n\ndef target_ids(proc, letter):\n    tok = proc.tokenizer\n    ids = tok.encode(letter, add_special_tokens=False)\n    assert len(ids) == 1, f"\'{letter}\'가 한 토큰이 아닙니다: {ids}"\n    im_end = tok.convert_tokens_to_ids("<|im_end|>")\n    assert isinstance(im_end, int) and im_end >= 0\n    return ids + [im_end]\n\n\ndef make_example(proc, row, image, consensus=False):\n    """추론과 같은 프롬프트 + [정답 글자, <|im_end|>]. 손실은 뒤의 두 토큰에만 겁니다."""\n    inputs = vr.encode(proc, vr.build_messages(row, image, consensus=consensus))\n    tgt = torch.tensor([target_ids(proc, row["answer"])], dtype=inputs["input_ids"].dtype)\n    n_prompt = inputs["input_ids"].shape[-1]\n    out = {}\n    for k, v in inputs.items():\n        if torch.is_tensor(v) and v.dim() == 2 and v.shape == (1, n_prompt):\n            fill = torch.ones_like(tgt) if k == "attention_mask" else (\n                tgt if k == "input_ids" else torch.zeros_like(tgt))\n            out[k] = torch.cat([v, fill.to(v.dtype)], dim=1)\n        else:\n            out[k] = v\n    return out, tgt\n\n\ndef answer_loss(model, batch, tgt, device="cuda:0"):\n    """정답 두 토큰에 대한 CE. 마지막 n+1개 위치의 로짓만 계산해 248K 어휘 전체 로짓을 피합니다.\n\n    위치 p의 로짓은 p+1 토큰을 예측하므로, 끝에서 n+1개를 남기면 앞의 n개가 정답 토큰을 예측합니다.\n    """\n    batch = {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in batch.items()}\n    tgt = tgt.to(device)\n    n = tgt.shape[-1]\n    logits = model(**batch, logits_to_keep=n + 1, use_cache=False).logits[:, :n].float()\n    return torch.nn.functional.cross_entropy(logits.reshape(-1, logits.shape[-1]), tgt.reshape(-1))\n\n\ndef attach_lora(model, r=16, alpha=32, dropout=0.05):\n    from peft import LoraConfig, get_peft_model\n\n    for p in model.parameters():\n        p.requires_grad_(False)\n    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})\n    model.enable_input_require_grads()\n    cfg = LoraConfig(r=r, lora_alpha=alpha, lora_dropout=dropout, bias="none",\n                     target_modules=LORA_TARGET, task_type="CAUSAL_LM")\n    model = get_peft_model(model, cfg)\n    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)\n    assert n_train > 0, "LoRA가 붙은 층이 없습니다. LORA_TARGET을 확인하세요"\n    return model\n\n\ndef cosine_lr(step, total, warmup):\n    if step < warmup:\n        return (step + 1) / warmup\n    progress = (step - warmup) / max(1, total - warmup)\n    return 0.5 * (1 + math.cos(math.pi * min(1.0, progress)))\n\n\ndef save_checkpoint(model, optimizer, scheduler, state, ckpt_dir):\n    """어댑터 + 옵티마이저 + 진행 위치. Windows 장시간 학습이 끊겨도 이어갈 수 있게 합니다."""\n    ckpt_dir = Path(ckpt_dir)\n    tmp = ckpt_dir.with_name(ckpt_dir.name + ".tmp")\n    model.save_pretrained(str(tmp))\n    torch.save({"optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict()},\n               tmp / "train_state.pt")\n    (tmp / "progress.json").write_text(json.dumps(state, indent=2), encoding="utf-8")\n    if ckpt_dir.exists():\n        import shutil\n        shutil.rmtree(ckpt_dir)\n    tmp.rename(ckpt_dir)\n\n\ndef load_checkpoint(model, optimizer, scheduler, ckpt_dir):\n    from peft import set_peft_model_state_dict\n    from safetensors.torch import load_file\n\n    ckpt_dir = Path(ckpt_dir)\n    set_peft_model_state_dict(model, load_file(str(ckpt_dir / "adapter_model.safetensors")))\n    st = torch.load(ckpt_dir / "train_state.pt", map_location="cpu", weights_only=False)\n    optimizer.load_state_dict(st["optimizer"])\n    scheduler.load_state_dict(st["scheduler"])\n    return json.loads((ckpt_dir / "progress.json").read_text(encoding="utf-8"))\n\n\ndef train_loop(model, proc, order, *, grad_accum, lr, warmup_ratio, max_grad_norm, save_every,\n               ckpt_dir, log_path, consensus=False, resume=True):\n    """문항 하나씩 손실을 쌓아 grad_accum마다 갱신합니다. save_every 스텝마다 ckpt_dir에 저장하고,\n    다시 부르면 마지막 체크포인트에서 이어 갑니다. Colab 세션이 끊겨도 Drive의 ckpt_dir로 재개합니다.\n    """\n    import time\n    import common_eval as _ce\n\n    total = len(order) // grad_accum\n    warmup = max(1, int(total * warmup_ratio))\n    params = [p for p in model.parameters() if p.requires_grad]\n    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=0.0)\n    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda s: cosine_lr(s, total, warmup))\n    ckpt_dir, log_path = Path(ckpt_dir), Path(log_path)\n    start = 0\n    if resume and (ckpt_dir / "progress.json").is_file():\n        start = load_checkpoint(model, optimizer, scheduler, ckpt_dir)["step"]\n        print(f"체크포인트에서 이어서: {start}/{total} 스텝")\n    elif log_path.is_file():\n        log_path.unlink()\n    model.train()\n    t_start = time.time()\n    with log_path.open("a", encoding="utf-8") as log:\n        for step in range(start, total):\n            t0 = time.time()\n            losses = []\n            for row in order[step * grad_accum:(step + 1) * grad_accum]:\n                batch, tgt = make_example(proc, row, _ce.open_image(row), consensus=consensus)\n                loss = answer_loss(model, batch, tgt)\n                (loss / grad_accum).backward()\n                losses.append(loss.item())\n                del batch, tgt, loss\n            grad_norm = torch.nn.utils.clip_grad_norm_(params, max_grad_norm)\n            optimizer.step(); scheduler.step(); optimizer.zero_grad(set_to_none=True)\n            item = dict(step=step + 1, loss=sum(losses) / len(losses), max_loss=max(losses),\n                        lr=scheduler.get_last_lr()[0], grad_norm=float(grad_norm),\n                        seconds=time.time() - t0)\n            log.write(json.dumps(item) + "\\n"); log.flush()\n            done = step + 1 - start\n            if done == 1 or (step + 1) % 10 == 0 or step + 1 == total:\n                left = (time.time() - t_start) / done * (total - step - 1)\n                print(f"  {step + 1}/{total}  loss {item[\'loss\']:.4f}  max {item[\'max_loss\']:.3f}  "\n                      f"{item[\'seconds\']:.0f}초/스텝  남은 약 {left / 60:.0f}분", flush=True)\n            if (step + 1) % save_every == 0 or step + 1 == total:\n                save_checkpoint(model, optimizer, scheduler, {"step": step + 1, "total": total}, ckpt_dir)\n    return total\n\n\ndef merge_and_save(base_dir, adapter_dir, out_dir, dtype=torch.bfloat16, device="cuda:0"):\n    """원본(bf16)에 어댑터를 합쳐 일반 체크포인트로 저장합니다. vLLM은 이 폴더를 그대로 올립니다.\n\n    vLLM이 Qwen3.5 linear attention 층의 LoRA를 직접 받는지 보장할 수 없어 합친 모델로 평가합니다.\n    4bit로 학습했어도 합치는 대상은 bf16 원본입니다.\n    """\n    from peft import PeftModel\n    from transformers import AutoProcessor, Qwen3_5ForConditionalGeneration\n\n    base = Qwen3_5ForConditionalGeneration.from_pretrained(\n        str(base_dir), dtype=dtype, device_map={"": device}, local_files_only=True)\n    merged = PeftModel.from_pretrained(base, str(adapter_dir)).merge_and_unload()\n    out_dir = Path(out_dir)\n    merged.save_pretrained(str(out_dir), safe_serialization=True, max_shard_size="5GB")\n    AutoProcessor.from_pretrained(str(base_dir), local_files_only=True).save_pretrained(str(out_dir))\n    del merged, base\n    copy_missing_weights(base_dir, out_dir)\n    return out_dir\n\n\ndef copy_missing_weights(base_dir, out_dir):\n    """원본에는 있는데 저장본에 없는 가중치(mtp.* 다중 토큰 예측 층)를 그대로 옮겨 담습니다.\n\n    transformers는 mtp 층을 불러오지 않아 save_pretrained 결과에서 빠집니다. 학습과 무관한 층이지만\n    config에는 남아 있어, vLLM이 찾을 경우를 대비해 원본과 같은 가중치 집합으로 맞춥니다.\n    """\n    from safetensors import safe_open\n    from safetensors.torch import save_file\n\n    base_dir, out_dir = Path(base_dir), Path(out_dir)\n    base_map = json.loads((base_dir / "model.safetensors.index.json").read_text())["weight_map"]\n    index_path = out_dir / "model.safetensors.index.json"\n    index = json.loads(index_path.read_text())\n    missing = sorted(set(base_map) - set(index["weight_map"]))\n    assert all(k.startswith("mtp.") for k in missing), f"예상 밖 누락: {[k for k in missing if not k.startswith(\'mtp.\')][:3]}"\n    if not missing:\n        return []\n    tensors = {}\n    for shard in sorted({base_map[k] for k in missing}):\n        with safe_open(str(base_dir / shard), framework="pt") as f:\n            for k in missing:\n                if base_map[k] == shard:\n                    tensors[k] = f.get_tensor(k)\n    save_file(tensors, str(out_dir / "model-extra-mtp.safetensors"), metadata={"format": "pt"})\n    for k in missing:\n        index["weight_map"][k] = "model-extra-mtp.safetensors"\n    index_path.write_text(json.dumps(index, indent=2), encoding="utf-8")\n    return missing\n\n\ndef _self_test():\n    import re\n    ok = "model.language_model.layers.3.linear_attn.in_proj_qkv"\n    assert re.fullmatch(LORA_TARGET, ok)\n    assert re.fullmatch(LORA_TARGET, "model.language_model.layers.7.mlp.down_proj")\n    assert not re.fullmatch(LORA_TARGET, "model.visual.blocks.0.attn.qkv")\n    assert not re.fullmatch(LORA_TARGET, "lm_head")\n    assert abs(cosine_lr(0, 100, 10) - 0.1) < 1e-9 and abs(cosine_lr(100, 100, 10)) < 1e-9\n    votes = lambda *v: dict(zip(DEV_VOTES, list(v) + [""] * (5 - len(v))))\n    assert dev_majority(votes("a", "a", "a", "b", "c")) == "a"\n    assert dev_majority(votes("a", "a", "b", "b", "c")) is None        # 2표 동률\n    assert dev_majority(votes("a", "a", "b", "c", "d")) is None        # 2표 단독이지만 3표 미만\n    assert dev_majority(votes("a", "a", "b", "c", "d"), min_votes=2) == "a"\n    assert dev_majority(votes()) is None\n    print("qlora OK")\n\n\nif __name__ == "__main__":\n    _self_test()\n', 'vllm_eval.py': '"""Colab vLLM 추론 공통 코드 (E7 이후).\n\nE4·E6 Colab 노트북은 이 코드를 셀에 복사해 두었습니다. E7부터는 노트북이 이 파일 원문을 품고\nimport합니다. 메시지·이미지 규칙·파서는 로컬 추론(vlm_runtime)과 같아야 하므로 common_eval을 씁니다.\n"""\nimport json\nimport math\nimport shutil\nimport time\nfrom math import comb\nfrom pathlib import Path\n\nimport pandas as pd\n\nimport common_eval as ce\n\nCHUNK = 256\n\n\ndef letter_token_ids(tokenizer):\n    return {ch: sorted({tokenizer.encode(f, add_special_tokens=False)[0]\n                        for f in (ch, " " + ch, ch.upper(), " " + ch.upper())\n                        if tokenizer.encode(f, add_special_tokens=False)})\n            for ch in ce.CHOICES}\n\n\ndef make_request(proc, row, consensus=False, image_fn=None):\n    """로컬과 같은 메시지를 HF 채팅 템플릿으로 렌더링하고, 이미지는 같은 open_image로 줄여 넘깁니다.\n\n    image_fn을 주면 open_image 대신 씁니다(E8 Zoom의 고해상도 입력).\n    """\n    image = (image_fn or ce.open_image)(row)\n    messages = [{"role": "system", "content": [{"type": "text", "text": ce.SYSTEM}]},\n                {"role": "user", "content": [{"type": "image"},\n                                             {"type": "text",\n                                              "text": ce.make_prompt(row, consensus=consensus)}]}]\n    prompt = proc.apply_chat_template(messages, tokenize=False, add_generation_prompt=True,\n                                      enable_thinking=False)\n    return {"prompt": prompt, "multi_modal_data": {"image": image}}, image.size\n\n\ndef letter_logprobs(first_step, letters):\n    """상위 k개 안에 없는 글자는 -1e9. JSON에 -inf를 쓰지 않으려고 유한값을 씁니다."""\n    out = {}\n    for ch, ids in letters.items():\n        vals = [first_step[i].logprob for i in ids if i in first_step]\n        out[ch] = max(vals) if vals else -1e9\n    return out\n\n\ndef run(llm, proc, sampling, rows, out_dir, drive_dir=None, consensus=False, desc="", image_fn=None):\n    """묶음 단위로 추론해 predictions.jsonl에 쓰고, 묶음마다 drive_dir로 복사합니다.\n\n    다시 부르면 기록된 문항을 건너뜁니다. out_dir에 기록이 없고 drive_dir에 있으면 먼저 가져옵니다.\n    """\n    out_dir = Path(out_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n    if drive_dir is not None:\n        drive_dir = Path(drive_dir)\n        drive_dir.mkdir(parents=True, exist_ok=True)\n    pred_path = out_dir / "predictions.jsonl"\n    if not pred_path.is_file() and drive_dir is not None and (drive_dir / "predictions.jsonl").is_file():\n        shutil.copy2(drive_dir / "predictions.jsonl", pred_path)\n        print("Drive에서 이전 기록을 복구했습니다")\n    done = set()\n    if pred_path.is_file():\n        done = {json.loads(l)["id"] for l in pred_path.open(encoding="utf-8") if l.strip()}\n    todo = [r for r in rows if r["id"] not in done]\n    print(f"[{desc or out_dir.name}] 대상 {len(rows)} / 기록됨 {len(done)} / 남은 {len(todo)}")\n\n    letters = letter_token_ids(proc.tokenizer)\n    instruction = ce.CONSENSUS_INSTRUCTION if consensus else ce.ANSWER_INSTRUCTION\n    t_all = time.time()\n    for start in range(0, len(todo), CHUNK):\n        batch = todo[start:start + CHUNK]\n        t0 = time.time()\n        reqs, sizes = zip(*(make_request(proc, r, consensus, image_fn) for r in batch))\n        if start == 0:\n            assert "<|image_pad|>" in reqs[0]["prompt"], "프롬프트에 이미지 토큰이 없습니다"\n            assert reqs[0]["prompt"].count(instruction) == 1, "지시문이 기대와 다릅니다"\n        outs = llm.generate(list(reqs), sampling, use_tqdm=False)\n        elapsed = time.time() - t0\n        with pred_path.open("a", encoding="utf-8") as log:\n            for row, size, o in zip(batch, sizes, outs):\n                c = o.outputs[0]\n                raw = c.text.strip()\n                log.write(json.dumps(dict(\n                    id=row["id"], raw_output=raw, pred=ce.parse_choice(raw),\n                    letter_logprobs=letter_logprobs(c.logprobs[0], letters) if c.logprobs else {},\n                    answer_truncated=c.finish_reason == "length", answer_tokens=len(c.token_ids),\n                    seconds=elapsed / len(batch), image_width=size[0], image_height=size[1]),\n                    ensure_ascii=False) + "\\n")\n        if drive_dir is not None:\n            for f in out_dir.glob("*"):\n                if f.is_file():\n                    shutil.copy2(f, drive_dir / f.name)\n        n_done = start + len(batch)\n        print(f"  {n_done}/{len(todo)}  {elapsed / len(batch):.3f}초/문항  "\n              f"남은 약 {(time.time() - t_all) / n_done * (len(todo) - n_done) / 60:.0f}분")\n    return read(pred_path)\n\n\ndef read(pred_path):\n    res = pd.read_json(pred_path, lines=True).drop_duplicates("id", keep="last")\n    res["conf"] = res.letter_logprobs.map(\n        lambda lp: math.exp(max(lp.values())) if isinstance(lp, dict) and lp else float("nan"))\n    return res\n\n\ndef fill_invalid(res):\n    """제출용. 파서가 실패한 문항은 기록된 a~d 로그확률 중 가장 높은 글자로 채웁니다."""\n    res = res.copy()\n    bad = res.pred == "INVALID"\n    res.loc[bad, "pred"] = res.loc[bad, "letter_logprobs"].map(\n        lambda lp: max(lp, key=lp.get) if lp else "INVALID")\n    return res, int(bad.sum())\n\n\ndef mcnemar_exact(b, c):\n    n = b + c\n    return 1.0 if n == 0 else min(1.0, 2 * sum(comb(n, k) for k in range(min(b, c) + 1)) / 2 ** n)\n\n\ndef paired(gold, pred_a, pred_b, name_a, name_b):\n    """gold(id, answer)에 두 예측(id, pred)을 맞대 McNemar를 출력합니다."""\n    j = (gold[["id", "answer"]]\n         .merge(pred_a[["id", "pred"]].rename(columns={"pred": "a"}), on="id")\n         .merge(pred_b[["id", "pred"]].rename(columns={"pred": "b"}), on="id"))\n    ok_a, ok_b = j.a == j.answer, j.b == j.answer\n    b, c = int((ok_a & ~ok_b).sum()), int((~ok_a & ok_b).sum())\n    n = len(j)\n    diff, se = (c - b) / n, math.sqrt(b + c) / n\n    p = mcnemar_exact(b, c)\n    print(f"[{n}문항] {name_a} {ok_a.mean():.4f} / {name_b} {ok_b.mean():.4f}")\n    print(f"  {name_a}만 정답 {b} / {name_b}만 정답 {c} / 차이 {diff * 100:+.2f}pp "\n          f"95%CI [{(diff - 1.96 * se) * 100:+.2f}, {(diff + 1.96 * se) * 100:+.2f}] / McNemar p = {p:.4g}")\n    return dict(n=n, a_only=b, b_only=c, diff=diff, p=p)\n\n\ndef _self_test():\n    assert abs(mcnemar_exact(11, 17) - 0.3449) < 1e-3\n    res = pd.DataFrame({"id": ["x", "y"], "pred": ["INVALID", "b"],\n                        "letter_logprobs": [{"a": -3.0, "b": -2.0, "c": -0.1, "d": -5.0}, {}]})\n    filled, n = fill_invalid(res)\n    assert n == 1 and list(filled.pred) == ["c", "b"]\n    print("vllm_eval OK")\n\n\nif __name__ == "__main__":\n    _self_test()\n', 'tta.py': '"""E8 추론 단계 기법(TTA): 선지 순서 4회 평균(Order4) → 불확실 문항 선별(Gate) → 고해상도 재평가(Zoom) → 결합(Blend).\n\n팀 0.97110 파이프라인(팀원, 팀 레포 `팀원 제출 패키지`의\nexperiment/scripts/mcq_core.py, output_zoom_order4_bf16_v1/qwen36_zoom_colab.py)과 같은 규칙입니다.\nGate 기준과 Blend 비율은 팀이 자기 검증 1,343문항으로 정한 값을 **고치지 않고** 씁니다.\nholdout에서 다시 고르면 holdout이 독립 확인 세트가 아니게 됩니다.\n\n확률은 a~d 네 글자의 로그확률을 소프트맥스한 상대 점수입니다. 보정된 정답 확률이 아닙니다.\n"""\nimport math\nfrom pathlib import Path\n\nimport pandas as pd\n\nimport common_eval as ce\n\n# 네 순환 순서. 모든 선지가 모든 자리에 정확히 한 번씩 놓입니다. perm[i]는 i번째 자리에 보이는 원래 선지입니다.\nORDERS = ("abcd", "bcda", "cdab", "dabc")\nIDENTITY = ORDERS[0]\n\nGATE_MAX_PROB = 0.85      # 평균 확률 최댓값이 이보다 낮으면 재평가\nGATE_MARGIN = 0.25        # 1·2위 차이가 이보다 작으면 재평가\nBLEND_WEIGHT = 0.5        # 재평가 확률의 비중\n\nZOOM_MIN_PIXELS = 1_474_560   # 작은 이미지는 이 넓이까지 키움 (팀 Zoom 입력과 같음)\nZOOM_MAX_PIXELS = 2_097_152   # 큰 이미지는 이 넓이까지 줄임. 비전 토큰 약 2,048개\n\nPROB_COLS = [f"p_{ch}" for ch in ce.CHOICES]\n\n\ndef permute_row(row, perm):\n    """perm 순서로 선지를 바꾼 문항 사본. 질문·이미지 경로·id는 그대로입니다."""\n    shown = dict(row)\n    for slot, orig in zip(ce.CHOICES, perm):\n        shown[slot] = row[orig]\n    return shown\n\n\ndef slot_probs(letter_logprobs):\n    """화면 자리 a~d의 로그확률을 네 글자 안에서 소프트맥스합니다. 기록이 없으면 None."""\n    if not isinstance(letter_logprobs, dict) or not letter_logprobs:\n        return None\n    vals = [letter_logprobs[ch] for ch in ce.CHOICES]\n    top = max(vals)\n    exps = [math.exp(v - top) for v in vals]\n    return [e / sum(exps) for e in exps]\n\n\ndef to_original(probs, perm):\n    """자리별 확률을 원래 선지 a~d 순서로 되돌립니다. 되돌리기는 한 번만 해야 합니다."""\n    out = [0.0] * len(ce.CHOICES)\n    for slot, orig in enumerate(perm):\n        out[ce.CHOICES.index(orig)] = probs[slot]\n    return out\n\n\ndef _pred(p):\n    return ce.CHOICES[max(range(len(p)), key=p.__getitem__)]\n\n\ndef summarize(probs, order_preds=None):\n    """원래 순서 확률 한 줄 → pred·conf·margin·순서 불일치."""\n    top, second = sorted(probs, reverse=True)[:2]\n    row = dict(zip(PROB_COLS, probs), pred=_pred(probs), conf=top, margin=top - second)\n    row["disagree"] = bool(order_preds) and len(set(order_preds)) > 1\n    return row\n\n\ndef order4(results):\n    """{perm: ve.read 결과} → 문항별 평균 확률 표. 네 순서가 모두 같은 문항을 가져야 합니다."""\n    ids = set(results[IDENTITY].id)\n    per_order = {}\n    for perm, res in results.items():\n        if set(res.id) != ids:\n            raise ValueError(f"{perm}: 문항 집합이 기준 순서와 다릅니다")\n        for i, lp in zip(res.id, res.letter_logprobs):\n            p = slot_probs(lp)\n            if p is None:\n                raise ValueError(f"{perm}/{i}: a~d 로그확률이 없습니다")\n            per_order.setdefault(i, []).append(to_original(p, perm))\n    rows = []\n    for i, ps in per_order.items():\n        mean = [sum(col) / len(ps) for col in zip(*ps)]\n        rows.append(dict(id=i, **summarize(mean, [_pred(p) for p in ps])))\n    return pd.DataFrame(rows)\n\n\ndef gate(df, max_prob=GATE_MAX_PROB, margin=GATE_MARGIN):\n    """재평가할 문항 표시. 팀 기준: 최댓값 < 0.85 또는 차이 < 0.25 또는 순서 불일치."""\n    return (df.conf < max_prob) | (df.margin < margin) | df.disagree\n\n\ndef blend(base, review, weight=BLEND_WEIGHT):\n    """review에 있는 문항만 base와 섞고 나머지는 base를 유지합니다."""\n    if not set(review.id) <= set(base.id):\n        raise ValueError("재평가 문항이 기준 표에 없습니다")\n    rev = review.set_index("id")[PROB_COLS]\n    rows = []\n    for rec in base.to_dict("records"):\n        p = [rec[c] for c in PROB_COLS]\n        if rec["id"] in rev.index:\n            p = [(1 - weight) * a + weight * b for a, b in zip(p, rev.loc[rec["id"]])]\n        rows.append(dict(id=rec["id"], reviewed=rec["id"] in rev.index, **summarize(p)))\n    return pd.DataFrame(rows)\n\n\ndef zoom_image(row, min_pixels=ZOOM_MIN_PIXELS, max_pixels=ZOOM_MAX_PIXELS):\n    """원본(960 제한 없이)을 넓이 [min_pixels, max_pixels] 안으로 맞춥니다. 작은 이미지는 키웁니다."""\n    from PIL import Image\n    with Image.open(ce.DATA_DIR / str(row["path"])) as im:\n        im = im.convert("RGB")\n        area = im.width * im.height\n        target = min(max(area, min_pixels), max_pixels)\n        if target != area:\n            scale = math.sqrt(target / area)\n            im = im.resize((max(1, round(im.width * scale)), max(1, round(im.height * scale))),\n                           Image.Resampling.LANCZOS)\n        return im\n\n\ndef vision_tokens(proc, image):\n    """HF 프로세서 기준 비전 토큰 수. Zoom이 실제로 입력을 키웠는지 확인할 때 씁니다."""\n    grid = proc.image_processor(images=[image], return_tensors="pt")["image_grid_thw"][0]\n    merge = getattr(proc.image_processor, "merge_size", 2)\n    return int(grid.prod()) // (merge * merge)\n\n\ndef run_orders(ve, llm, proc, sampling, rows, out_dir, drive_dir=None, image_fn=None, desc=""):\n    """네 순서를 차례로 추론해 순서별 폴더에 저장하고 {perm: 결과}를 돌려줍니다. 끊기면 이어 갑니다."""\n    results = {}\n    for perm in ORDERS:\n        shown = [permute_row(r, perm) for r in rows]\n        sub_drive = Path(drive_dir) / perm if drive_dir is not None else None\n        results[perm] = ve.run(llm, proc, sampling, shown, Path(out_dir) / perm, sub_drive,\n                               image_fn=image_fn, desc=f"{desc} {perm}")\n    return results\n\n\ndef _self_test():\n    row = {"id": "x", "a": "A", "b": "B", "c": "C", "d": "D"}\n    assert [permute_row(row, "bcda")[s] for s in ce.CHOICES] == ["B", "C", "D", "A"]\n    # \'bcda\'로 보여 주고 자리 d(원래 a)를 고르면 원래 a가 가장 높아야 합니다.\n    assert _pred(to_original([0.1, 0.1, 0.1, 0.7], "bcda")) == "a"\n    p = slot_probs({"a": -0.1, "b": -3.0, "c": -1e9, "d": -5.0})\n    assert abs(sum(p) - 1) < 1e-9 and p[2] == 0.0\n    df = pd.DataFrame([summarize([0.9, 0.05, 0.03, 0.02], list("aaaa")),\n                       summarize([0.9, 0.05, 0.03, 0.02], list("aaab")),\n                       summarize([0.6, 0.3, 0.05, 0.05], list("aaaa"))])\n    assert list(gate(df)) == [False, True, True]\n    base = pd.DataFrame([dict(id="x", **summarize([0.45, 0.40, 0.1, 0.05]))])\n    rev = pd.DataFrame([dict(id="x", **summarize([0.20, 0.70, 0.05, 0.05]))])\n    assert blend(base, rev).pred[0] == "b"\n    print("tta OK")\n\n\nif __name__ == "__main__":\n    _self_test()\n'}
SOURCE_SHA = {'common_eval.py': 'ff6dc88f6916fa149a3a8fdec7dfa60fa384701c542aa19fedc582ce6f8850ae', 'vlm_runtime.py': 'e6735a19ea96a3c768bba136c8c36de7ec76efaa9494c0c8b6c955e97555bfd0', 'qlora.py': '56f954f09db990bebadb03b7165c5227bef34e257add3c7f9d17dc5100e16a59', 'vllm_eval.py': '5e1de9bbce0dea833ca97fbfb4f50787a612a9d52480baff17ed1c16d35bacf3', 'tta.py': '6cca767977cf4c9d689e07992900489f2ca3fcb04899f056ca77ce76eecfb9ce'}
import hashlib
SRC_DIR = Path("/content/src"); SRC_DIR.mkdir(parents=True, exist_ok=True)
for name, text in SOURCES.items():
    assert hashlib.sha256(text.encode("utf-8")).hexdigest() == SOURCE_SHA[name], name
    (SRC_DIR / name).write_text(text, encoding="utf-8")
sys.path.insert(0, str(SRC_DIR))
import common_eval as ce
import vlm_runtime as vr
import qlora
import vllm_eval as ve
import tta

ce.DATA_DIR = DATA_ROOT
ce._self_test(); ve._self_test(); tta._self_test()
print("데이터:", DATA_ROOT)

## 3. 설정

In [ ]:
CONFIG = "C"
SOURCE_RUN = f"exp_E7_27b_{CONFIG}"
RUN_NAME = f"exp_E8_tta_{CONFIG}"
DRIVE_RUN = DRIVE_BASE / "output" / RUN_NAME
ADAPTER_DIR = DRIVE_BASE / "output" / SOURCE_RUN / "adapter"
BASE_DIR = Path("/content/models/Qwen3.5-27B")
MERGED_DIR = Path("/content/merged") / SOURCE_RUN
LOCAL_OUT = Path("/content/out") / RUN_NAME
DRIVE_RUN.mkdir(parents=True, exist_ok=True)
assert (ADAPTER_DIR / "adapter_config.json").is_file(), f"E7 어댑터가 없습니다: {ADAPTER_DIR}"
print("실행 이름:", RUN_NAME, "/ 어댑터:", ADAPTER_DIR)
print(f"Gate: 최댓값 < {tta.GATE_MAX_PROB} 또는 차이 < {tta.GATE_MARGIN} 또는 순서 불일치 / Blend {tta.BLEND_WEIGHT}")
print(f"Zoom 넓이: {tta.ZOOM_MIN_PIXELS:,} ~ {tta.ZOOM_MAX_PIXELS:,} 픽셀 (기본 입력은 긴 변 {ce.IMAGE_MAX_EDGE})")

## 4. holdout 1,300

E6에서 나눈 holdout입니다. E7 학습에 쓰지 않았고, 설정 선택에도 쓰지 않았습니다.

In [ ]:
HOLDOUT_NUMS = [37, 57, 61, 65, 72, 77, 78, 84, 91, 93, 106, 108, 110, 111, 113, 117, 119, 124, 127, 130, 132, 134, 143, 145, 155, 156, 166, 169, 174, 177, 185, 192, 193, 203, 205, 208, 209, 213, 214, 221, 224, 225, 226, 230, 231, 234, 238, 239, 241, 245, 247, 251, 252, 260, 266, 267, 271, 273, 277, 280, 281, 290, 299, 300, 303, 304, 307, 309, 312, 316, 317, 319, 322, 323, 325, 335, 339, 346, 347, 355, 356, 360, 363, 365, 370, 371, 381, 387, 388, 390, 401, 428, 432, 442, 463, 475, 480, 481, 484, 500, 505, 508, 515, 517, 519, 524, 528, 529, 531, 537, 542, 551, 558, 567, 569, 570, 573, 574, 575, 584, 594, 599, 601, 603, 613, 614, 625, 641, 649, 650, 652, 670, 673, 675, 698, 703, 718, 723, 742, 746, 756, 758, 765, 775, 776, 778, 785, 786, 787, 792, 797, 803, 807, 827, 829, 833, 842, 846, 849, 858, 863, 870, 871, 873, 875, 882, 893, 909, 912, 913, 937, 939, 940, 944, 949, 968, 979, 983, 985, 1000, 1003, 1005, 1006, 1012, 1018, 1022, 1033, 1034, 1039, 1042, 1045, 1048, 1049, 1052, 1061, 1065, 1070, 1072, 1075, 1079, 1080, 1083, 1087, 1092, 1094, 1097, 1106, 1116, 1117, 1118, 1119, 1120, 1123, 1127, 1134, 1139, 1145, 1149, 1150, 1152, 1155, 1156, 1166, 1171, 1172, 1174, 1180, 1181, 1184, 1193, 1197, 1199, 1200, 1201, 1211, 1213, 1218, 1221, 1223, 1224, 1230, 1240, 1272, 1274, 1275, 1281, 1282, 1287, 1289, 1297, 1298, 1315, 1324, 1325, 1328, 1329, 1333, 1339, 1341, 1342, 1348, 1352, 1361, 1368, 1372, 1373, 1383, 1390, 1391, 1392, 1402, 1404, 1405, 1408, 1420, 1421, 1423, 1429, 1433, 1434, 1437, 1454, 1457, 1458, 1460, 1461, 1471, 1475, 1483, 1500, 1512, 1520, 1525, 1528, 1530, 1531, 1533, 1535, 1537, 1539, 1540, 1543, 1545, 1555, 1560, 1563, 1566, 1578, 1580, 1583, 1593, 1599, 1604, 1608, 1609, 1614, 1615, 1621, 1626, 1641, 1649, 1650, 1651, 1659, 1663, 1670, 1682, 1700, 1705, 1713, 1725, 1729, 1730, 1733, 1734, 1737, 1744, 1752, 1753, 1754, 1755, 1758, 1762, 1770, 1771, 1772, 1781, 1802, 1803, 1811, 1824, 1825, 1826, 1827, 1828, 1837, 1860, 1861, 1863, 1867, 1871, 1874, 1884, 1888, 1891, 1892, 1893, 1894, 1896, 1903, 1906, 1910, 1913, 1914, 1919, 1923, 1929, 1930, 1931, 1933, 1946, 1954, 1960, 1968, 1973, 1974, 1975, 1977, 1981, 1982, 1987, 1988, 1991, 1992, 1995, 1996, 1997, 2005, 2009, 2012, 2014, 2018, 2031, 2041, 2046, 2049, 2060, 2065, 2078, 2079, 2085, 2087, 2091, 2094, 2102, 2105, 2107, 2120, 2131, 2132, 2133, 2144, 2149, 2150, 2152, 2161, 2162, 2166, 2167, 2172, 2173, 2177, 2180, 2189, 2192, 2198, 2206, 2210, 2214, 2216, 2222, 2229, 2233, 2241, 2263, 2268, 2270, 2273, 2286, 2288, 2300, 2308, 2313, 2315, 2324, 2328, 2344, 2346, 2349, 2352, 2353, 2356, 2363, 2364, 2366, 2368, 2384, 2385, 2390, 2415, 2430, 2436, 2441, 2445, 2449, 2450, 2451, 2455, 2458, 2461, 2467, 2468, 2470, 2475, 2481, 2482, 2484, 2485, 2487, 2491, 2498, 2510, 2514, 2515, 2517, 2519, 2522, 2527, 2536, 2538, 2546, 2548, 2559, 2561, 2564, 2565, 2581, 2583, 2586, 2587, 2590, 2610, 2623, 2625, 2628, 2629, 2652, 2658, 2659, 2662, 2668, 2669, 2671, 2691, 2697, 2700, 2704, 2706, 2710, 2721, 2725, 2730, 2731, 2736, 2741, 2743, 2749, 2757, 2771, 2772, 2777, 2782, 2784, 2786, 2797, 2798, 2810, 2812, 2814, 2815, 2816, 2817, 2830, 2836, 2842, 2852, 2864, 2865, 2866, 2877, 2886, 2897, 2901, 2907, 2909, 2916, 2925, 2928, 2931, 2936, 2937, 2943, 2967, 2972, 2976, 2977, 2993, 3003, 3007, 3011, 3013, 3020, 3045, 3049, 3054, 3071, 3073, 3076, 3082, 3083, 3086, 3100, 3113, 3115, 3120, 3121, 3128, 3147, 3154, 3170, 3189, 3190, 3194, 3199, 3202, 3208, 3213, 3221, 3228, 3230, 3239, 3242, 3244, 3250, 3266, 3275, 3280, 3281, 3289, 3295, 3298, 3305, 3311, 3314, 3323, 3328, 3333, 3338, 3349, 3359, 3361, 3363, 3364, 3368, 3374, 3383, 3397, 3398, 3408, 3411, 3415, 3433, 3441, 3447, 3448, 3451, 3452, 3455, 3458, 3459, 3470, 3489, 3490, 3512, 3518, 3540, 3541, 3547, 3548, 3549, 3556, 3560, 3562, 3569, 3580, 3582, 3583, 3589, 3594, 3599, 3616, 3622, 3623, 3628, 3632, 3658, 3663, 3665, 3673, 3676, 3680, 3682, 3688, 3692, 3696, 3698, 3699, 3718, 3724, 3732, 3738, 3742, 3744, 3746, 3757, 3767, 3768, 3781, 3792, 3794, 3796, 3802, 3807, 3809, 3821, 3832, 3847, 3848, 3855, 3867, 3875, 3888, 3890, 3899, 3910, 3911, 3912, 3916, 3920, 3930, 3937, 3939, 3943, 3947, 3952, 3953, 3973, 3977, 3979, 3981, 3985, 3992, 3996, 4001, 4002, 4004, 4015, 4018, 4030, 4033, 4035, 4039, 4044, 4050, 4059, 4063, 4064, 4066, 4068, 4071, 4074, 4082, 4106, 4108, 4116, 4117, 4122, 4125, 4128, 4130, 4132, 4141, 4149, 4150, 4163, 4165, 4167, 4186, 4190, 4191, 4192, 4193, 4194, 4195, 4196, 4202, 4204, 4208, 4210, 4213, 4214, 4222, 4223, 4225, 4229, 4230, 4237, 4241, 4246, 4254, 4257, 4258, 4264, 4267, 4274, 4287, 4288, 4291, 4293, 4297, 4304, 4305, 4306, 4310, 4315, 4320, 4332, 4338, 4339, 4341, 4354, 4360, 4366, 4367, 4372, 4376, 4379, 4380, 4384, 4391, 4392, 4395, 4398, 4401, 4402, 4404, 4405, 4408, 4409, 4415, 4419, 4430, 4439, 4445, 4449, 4450, 4451, 4452, 4454, 4456, 4461, 4462, 4468, 4471, 4478, 4481, 4485, 4488, 4491, 4497, 4509, 4510, 4518, 4522, 4528, 4531, 4539, 4540, 4553, 4555, 4569, 4574, 4580, 4584, 4595, 4596, 4600, 4602, 4603, 4605, 4609, 4614, 4625, 4630, 4635, 4638, 4639, 4640, 4647, 4648, 4649, 4657, 4659, 4665, 4668, 4670, 4673, 4675, 4682, 4683, 4701, 4705, 4708, 4717, 4719, 4720, 4721, 4722, 4723, 4729, 4736, 4738, 4742, 4744, 4748, 4749, 4753, 4754, 4755, 4759, 4760, 4764, 4771, 4780, 4786, 4789, 4790, 4792, 4793, 4794, 4800, 4801, 4802, 4806, 4808, 4809, 4812, 4814, 4815, 4822, 4824, 4826, 4827, 4831, 4836, 4840, 4843, 4847, 4848, 4849, 4854, 4855, 4859, 4864, 4866, 4868, 4874, 4878, 4883, 4889, 4907, 4913, 4914, 4917, 4918, 4925, 4926, 4944, 4948, 4953, 4954, 4957, 4968, 4971, 4985, 5009, 5013, 5015, 5031, 5036, 5045, 5049, 5058, 5077, 5080, 5082, 5086, 5090, 5104, 5108, 5109, 5116, 5121, 5126, 5130, 5145, 5148, 5151, 5155, 5158, 5164, 5167, 5169, 5170, 5171, 5179, 5180, 5181, 5182, 5185, 5187, 5188, 5200, 5209, 5217, 5220, 5223, 5224, 5245, 5250, 5254, 5261, 5266, 5268, 5269, 5270, 5271, 5279, 5281, 5285, 5289, 5290, 5308, 5315, 5316, 5317, 5319, 5331, 5340, 5344, 5361, 5371, 5374, 5378, 5381, 5383, 5384, 5388, 5394, 5397, 5399, 5403, 5420, 5422, 5429, 5441, 5445, 5447, 5452, 5456, 5459, 5461, 5464, 5467, 5478, 5483, 5486, 5488, 5495, 5499, 5505, 5506, 5509, 5517, 5518, 5519, 5521, 5524, 5525, 5526, 5532, 5534, 5536, 5540, 5541, 5542, 5545, 5547, 5555, 5556, 5557, 5560, 5564, 5572, 5577, 5593, 5596, 5601, 5605, 5610, 5618, 5622, 5629, 5637, 5639, 5653, 5657, 5682, 5683, 5684, 5693, 5715, 5723, 5729, 5730, 5733, 5747, 5758, 5775, 5776, 5789, 5790, 5798, 5809, 5811, 5818, 5820, 5821, 5828, 5832, 5833, 5844, 5845, 5855, 5857, 5861, 5862, 5867, 5874, 5875, 5880, 5885, 5888, 5890, 5895, 5904, 5906, 5909, 5911, 5915, 5927, 5928, 5931, 5936, 5941, 5942, 5943, 5950, 5956, 5962, 5970, 5972, 5981, 5982, 5983, 5992, 6001, 6008, 6017, 6021, 6025, 6031, 6034, 6037, 6040, 6050, 6052, 6059, 6065, 6067, 6069, 6072, 6080, 6084, 6100, 6101, 6113, 6114, 6129, 6131, 6138, 6143, 6146, 6149, 6151, 6154, 6157, 6160, 6162, 6168, 6169, 6171, 6179, 6182, 6189, 6196, 6200, 6204, 6211, 6212, 6216, 6217, 6223, 6241, 6250, 6259, 6260, 6265, 6267, 6269, 6283, 6284, 6287, 6288, 6290, 6302, 6308, 6315, 6323, 6340, 6347, 6352, 6359, 6371, 6374, 6375, 6377, 6378, 6390, 6398, 6404, 6407, 6413, 6416, 6418, 6422, 6426, 6427, 6438, 6445, 6461, 6464, 6476, 6477, 6487, 6488, 6489, 6490, 6495, 6503, 6509, 6522, 6528, 6531, 6537, 6544, 6547, 6548, 6549, 6551, 6556, 6567, 6570, 6571, 6576, 6581, 6582, 6584, 6586, 6588, 6590, 6594, 6603, 6607, 6611, 6613, 6618, 6631, 6633, 6635, 6637, 6643, 6645, 6647, 6652, 6663, 6664, 6667, 6671, 6672, 6673, 6674, 6675, 6679, 6684, 6690, 6693, 6702, 6703, 6707]
VAL672_NUMS = [37, 57, 65, 84, 91, 108, 113, 119, 124, 130, 132, 134, 145, 169, 174, 177, 193, 213, 221, 224, 225, 247, 251, 267, 273, 281, 300, 307, 309, 316, 317, 322, 356, 360, 363, 370, 388, 390, 401, 428, 432, 442, 481, 500, 508, 517, 528, 537, 551, 558, 567, 584, 594, 601, 603, 613, 641, 649, 650, 670, 673, 675, 698, 703, 756, 775, 778, 785, 787, 792, 797, 803, 807, 827, 833, 842, 846, 858, 863, 870, 871, 873, 893, 909, 912, 913, 937, 944, 968, 983, 985, 1000, 1005, 1006, 1012, 1034, 1039, 1042, 1045, 1049, 1061, 1072, 1079, 1080, 1092, 1094, 1097, 1106, 1117, 1118, 1120, 1127, 1139, 1145, 1149, 1152, 1155, 1156, 1166, 1171, 1172, 1193, 1199, 1200, 1201, 1213, 1272, 1275, 1297, 1298, 1324, 1325, 1328, 1329, 1333, 1341, 1342, 1348, 1361, 1372, 1392, 1402, 1420, 1433, 1437, 1458, 1461, 1471, 1483, 1512, 1525, 1528, 1530, 1533, 1537, 1543, 1560, 1578, 1593, 1614, 1641, 1650, 1659, 1663, 1705, 1725, 1730, 1734, 1744, 1753, 1755, 1758, 1762, 1770, 1803, 1825, 1827, 1828, 1837, 1861, 1867, 1888, 1891, 1892, 1893, 1894, 1903, 1919, 1930, 1931, 1954, 1960, 1977, 1981, 1982, 1992, 1997, 2014, 2018, 2031, 2046, 2049, 2060, 2079, 2085, 2087, 2105, 2120, 2133, 2144, 2149, 2172, 2173, 2177, 2189, 2198, 2206, 2216, 2222, 2241, 2263, 2288, 2308, 2313, 2315, 2324, 2344, 2346, 2349, 2352, 2353, 2356, 2363, 2364, 2366, 2368, 2384, 2415, 2436, 2449, 2458, 2467, 2470, 2481, 2487, 2498, 2514, 2517, 2527, 2538, 2546, 2548, 2561, 2565, 2583, 2586, 2590, 2628, 2662, 2668, 2669, 2697, 2704, 2725, 2730, 2731, 2736, 2741, 2743, 2749, 2771, 2777, 2784, 2797, 2812, 2816, 2817, 2830, 2836, 2852, 2864, 2865, 2907, 2925, 2928, 2931, 2937, 2967, 2972, 2993, 3003, 3011, 3013, 3045, 3054, 3073, 3076, 3083, 3086, 3100, 3113, 3115, 3121, 3128, 3154, 3194, 3199, 3202, 3228, 3230, 3239, 3244, 3250, 3266, 3280, 3289, 3298, 3323, 3328, 3333, 3349, 3359, 3361, 3364, 3374, 3383, 3408, 3441, 3447, 3448, 3451, 3452, 3455, 3458, 3470, 3489, 3541, 3549, 3580, 3582, 3583, 3594, 3599, 3616, 3622, 3623, 3632, 3658, 3663, 3665, 3673, 3676, 3680, 3688, 3696, 3698, 3699, 3718, 3757, 3792, 3802, 3809, 3821, 3832, 3875, 3888, 3890, 3899, 3912, 3916, 3920, 3930, 3943, 3947, 3973, 3977, 3981, 3985, 3996, 4018, 4039, 4044, 4068, 4074, 4082, 4106, 4117, 4125, 4130, 4132, 4149, 4167, 4186, 4191, 4193, 4195, 4204, 4208, 4213, 4229, 4254, 4257, 4264, 4267, 4291, 4293, 4297, 4305, 4306, 4310, 4315, 4332, 4341, 4354, 4367, 4372, 4380, 4392, 4398, 4401, 4402, 4404, 4408, 4409, 4419, 4430, 4439, 4445, 4450, 4454, 4461, 4468, 4471, 4478, 4491, 4497, 4518, 4528, 4531, 4540, 4553, 4569, 4584, 4596, 4600, 4602, 4603, 4605, 4614, 4630, 4639, 4649, 4657, 4659, 4668, 4670, 4701, 4708, 4719, 4720, 4744, 4748, 4754, 4760, 4764, 4771, 4786, 4789, 4790, 4801, 4802, 4808, 4812, 4814, 4822, 4824, 4831, 4836, 4843, 4847, 4859, 4866, 4868, 4874, 4878, 4889, 4914, 4917, 4918, 4925, 4944, 4954, 4968, 4971, 5009, 5013, 5045, 5058, 5077, 5080, 5109, 5126, 5145, 5148, 5155, 5158, 5167, 5169, 5182, 5187, 5188, 5209, 5220, 5245, 5254, 5261, 5266, 5268, 5269, 5285, 5289, 5290, 5308, 5331, 5340, 5361, 5371, 5374, 5384, 5420, 5422, 5452, 5459, 5461, 5478, 5486, 5488, 5495, 5499, 5506, 5509, 5518, 5519, 5521, 5524, 5525, 5526, 5532, 5534, 5536, 5545, 5547, 5555, 5556, 5560, 5564, 5610, 5618, 5622, 5629, 5653, 5683, 5684, 5723, 5729, 5730, 5747, 5789, 5790, 5811, 5818, 5832, 5833, 5844, 5845, 5855, 5857, 5861, 5867, 5880, 5885, 5888, 5895, 5904, 5906, 5928, 5936, 5950, 5962, 5972, 5982, 5992, 6017, 6025, 6031, 6037, 6040, 6050, 6069, 6100, 6101, 6113, 6131, 6143, 6149, 6154, 6157, 6160, 6162, 6169, 6171, 6179, 6189, 6196, 6212, 6216, 6217, 6241, 6259, 6260, 6267, 6283, 6284, 6287, 6288, 6290, 6302, 6308, 6315, 6323, 6340, 6347, 6359, 6371, 6375, 6378, 6390, 6404, 6413, 6416, 6418, 6422, 6426, 6427, 6438, 6445, 6464, 6476, 6522, 6528, 6547, 6548, 6549, 6551, 6556, 6576, 6584, 6590, 6603, 6611, 6618, 6637, 6645, 6647, 6663, 6671, 6673, 6675, 6684, 6693, 6702, 6703]
HOLDOUT = {f"train_{n:04d}.jpg" for n in HOLDOUT_NUMS}
VAL672 = {f"train_{n:04d}.jpg" for n in VAL672_NUMS}
assert len(HOLDOUT) == 1300 and VAL672 <= HOLDOUT
train_df = pd.read_csv(DATA_ROOT / "train.csv", keep_default_na=False)
gold = train_df[train_df.id.isin(HOLDOUT)].reset_index(drop=True)
holdout_rows = gold.drop(columns=["answer"]).to_dict("records")
print("holdout", len(gold), "/ 정답 분포", gold.answer.value_counts().sort_index().to_dict())

## 5. 원본 다운로드 · 병합

E7 어댑터를 bf16 원본에 합쳐 로컬 디스크에 저장합니다(약 54 GiB). 병합본은 Drive에 올리지 않습니다.

In [ ]:
from huggingface_hub import snapshot_download

if not (BASE_DIR / "config.json").is_file():
    snapshot_download(repo_id="Qwen/Qwen3.5-27B", local_dir=str(BASE_DIR))
vr.check_weights(BASE_DIR)
if (MERGED_DIR / "config.json").is_file():
    print("이미 병합됨:", MERGED_DIR)
else:
    t0 = time.time()
    qlora.merge_and_save(BASE_DIR, ADAPTER_DIR, MERGED_DIR, dtype=torch.bfloat16)
    gc.collect(); torch.cuda.empty_cache()
    print(f"병합 {time.time() - t0:.0f}초 ->", MERGED_DIR)

## 6. vLLM 적재 · Zoom 입력 확인

**메모리 오류가 나면** *세션 다시 시작* 후 2·3·4번 셀과 이 셀부터 실행하세요. 병합본은 디스크에 남아 있습니다.

답은 생성한 글자가 아니라 첫 토큰의 a~d 로그확률로 정하므로 `max_tokens=1`입니다
(그래서 기록의 `answer_truncated`는 항상 True이며 의미가 없습니다).
Zoom이 실제로 비전 토큰을 늘리는지 holdout 몇 장으로 확인하고, 늘지 않으면 멈춥니다.

In [ ]:
from transformers import AutoProcessor
from vllm import LLM, SamplingParams

llm = LLM(model=str(MERGED_DIR), dtype="bfloat16", max_model_len=4096, gpu_memory_utilization=0.90,
          limit_mm_per_prompt={"image": 1}, seed=42)
proc = AutoProcessor.from_pretrained(str(MERGED_DIR))
SAMPLING = SamplingParams(temperature=0.0, max_tokens=1, logprobs=20, seed=42)

for row in holdout_rows[:5]:
    native, zoom = ce.open_image(row), tta.zoom_image(row)
    n_tok, z_tok = tta.vision_tokens(proc, native), tta.vision_tokens(proc, zoom)
    print(f"{row['id']}: 기본 {native.size} {n_tok}토큰 -> Zoom {zoom.size} {z_tok}토큰")
    assert z_tok > n_tok, "Zoom이 비전 토큰을 늘리지 않습니다. 프로세서 max_pixels를 확인하세요"

## 7. E8-1 — holdout Order4

`abcd` 순서는 E7 단독과 같은 입력입니다. 먼저 E7 기록과 답이 같은지(재현) 확인한 뒤 Order4와 짝 비교합니다.

In [ ]:
def paired_all(stages, ids=None):
    """이웃 단계끼리, 그리고 첫 단계 대비 마지막 단계를 짝 비교합니다."""
    g = gold if ids is None else gold[gold.id.isin(ids)]
    names = list(stages)
    out = {}
    for a, b in list(zip(names, names[1:])) + ([(names[0], names[-1])] if len(names) > 2 else []):
        out[f"{a} -> {b}"] = ve.paired(g, stages[a], stages[b], a, b)
    return out


h_orders = tta.run_orders(ve, llm, proc, SAMPLING, holdout_rows, LOCAL_OUT / "holdout",
                          DRIVE_RUN / "holdout", desc="holdout")
h_o4 = tta.order4(h_orders)
h_base = h_orders[tta.IDENTITY].assign(pred=lambda d: d.letter_logprobs.map(
    lambda lp: tta.summarize(tta.slot_probs(lp))["pred"]))

prev_path = DRIVE_BASE / "output" / SOURCE_RUN / "holdout" / "predictions.jsonl"
if prev_path.is_file():
    prev = ve.read(prev_path)
    same = h_base[["id", "pred"]].merge(prev[["id", "pred"]], on="id", suffixes=("", "_e7"))
    print(f"재현: E7-{CONFIG} 기록과 abcd 답 일치 {(same.pred == same.pred_e7).mean():.4f} ({len(same)}문항)")
print(f"순서 불일치 문항 {int(h_o4.disagree.sum())} ({h_o4.disagree.mean():.1%})")
h_o4.to_csv(DRIVE_RUN / "holdout_order4.csv", index=False)
r1 = paired_all({f"E7-{CONFIG}": h_base, "Order4": h_o4})

## 8. E8-2 — Gate → Zoom → Blend

Gate에 걸린 문항만 고해상도로 다시 Order4하고, 기존 Order4와 반씩 섞습니다. 나머지 문항은 Order4를 유지합니다.

In [ ]:
def zoom_blend(o4, rows, split):
    """Gate 선별 → Zoom Order4 → Blend. (최종 표, Zoom 표)를 돌려줍니다."""
    picked = set(o4[tta.gate(o4)].id)
    zoom_rows = [r for r in rows if r["id"] in picked]
    print(f"[{split}] Gate 대상 {len(zoom_rows)} / {len(rows)} ({len(zoom_rows) / len(rows):.1%})")
    z_orders = tta.run_orders(ve, llm, proc, SAMPLING, zoom_rows, LOCAL_OUT / f"{split}_zoom",
                              DRIVE_RUN / f"{split}_zoom", image_fn=tta.zoom_image, desc=f"{split} zoom")
    z = z_orders[tta.IDENTITY]
    assert (z.image_width * z.image_height >= tta.ZOOM_MIN_PIXELS * 0.99).all(), "Zoom 입력이 커지지 않았습니다"
    zoom = tta.order4(z_orders)
    return tta.blend(o4, zoom), zoom


h_final, h_zoom = zoom_blend(h_o4, holdout_rows, "holdout")
h_final.to_csv(DRIVE_RUN / "holdout_final.csv", index=False)
stages = {f"E7-{CONFIG}": h_base, "Order4": h_o4, "Order4+Zoom": h_final}
r2 = paired_all(stages)
gated = set(h_final[h_final.reviewed].id)
print(f"\nGate 대상 {len(gated)}문항 안에서:")
g_in = gold[gold.id.isin(gated)]
ve.paired(g_in, h_o4, h_final, "Order4", "Order4+Zoom")

e6_path = DRIVE_BASE / "output" / "exp_E6_zeroshot_train" / "predictions.jsonl"
if e6_path.is_file():
    print("\n27B 제로샷(E6) 대비:")
    ve.paired(gold, ve.read(e6_path), h_final, "E6 제로샷", "E8 최종")
report = dict(config=CONFIG, gate=dict(max_prob=tta.GATE_MAX_PROB, margin=tta.GATE_MARGIN),
              blend=tta.BLEND_WEIGHT, zoom_pixels=[tta.ZOOM_MIN_PIXELS, tta.ZOOM_MAX_PIXELS],
              gated=len(gated), holdout={k: v for k, v in {**r1, **r2}.items()})
(DRIVE_RUN / "holdout_report.json").write_text(json.dumps(report, ensure_ascii=False, indent=2))
print("\n저장:", DRIVE_RUN / "holdout_report.json")

## 8-1. E8-A와 비교 — 제출 여부 판정

미리 정한 기준(`docs/decisions.md`)으로 판정합니다. 결과를 본 뒤 기준을 바꾸지 마세요.

- **확인 지점 ①** E7-C(기본 순서 1회)가 E7-A보다 뚜렷하게 나쁘면(E7-A만 정답이 더 많고 p < 0.1) 멈춤
- **확인 지점 ②** E8-C 최종 정확도가 E8-A(97.23%)보다 낮지 않으면 9번 제출, 낮으면 E8-A(0.97140) 유지

E8-A 오답 분석에서 학습으로 고칠 수 있다고 본 15문항(사람 해석 · 왜곡된 전사 선지)과
E8-A 오답 36문항 전체가 얼마나 고쳐졌는지도 따로 셉니다.

In [ ]:
TARGET = {f"train_{n:04d}.jpg" for n in [4748, 1368, 2288, 1230, 5811, 968, 3560, 4603, 3082, 5464, 2936, 5036, 1116, 3867, 1593]}
E8A_ERRORS = {f"train_{n:04d}.jpg" for n in [4748, 5890, 1368, 2485, 6488, 1092, 2288, 4522, 4801, 365, 2515, 1230, 5811, 968, 2210, 2662, 944, 4755, 6211, 5187, 3560, 4603, 937, 5637, 3082, 3899, 5464, 2936, 5036, 1116, 6702, 6487, 3867, 1593, 3013, 570]}
E7A_PATH = DRIVE_BASE / "output" / "exp_E7_27b_A" / "holdout" / "predictions.jsonl"
E8A_PATH = DRIVE_BASE / "output" / "exp_E8_tta_A" / "holdout_final.csv"
CHECK1_P = 0.1


def correct_in(res, ids):
    ans = gold.set_index("id").answer.reindex(sorted(ids))
    return int((res.set_index("id").pred.reindex(ans.index) == ans).sum())


if E7A_PATH.is_file():
    c1 = ve.paired(gold, ve.read(E7A_PATH), h_base, "E7-A", f"E7-{CONFIG}")
    worse = c1["a_only"] > c1["b_only"] and c1["p"] < CHECK1_P
    print(f"확인 지점 ①: {'멈춤 (E7-' + CONFIG + '이 뚜렷하게 나쁨)' if worse else '통과'}\n")
if E8A_PATH.is_file():
    e8a = pd.read_csv(E8A_PATH)
    c2 = ve.paired(gold, e8a, h_final, "E8-A", f"E8-{CONFIG}")
    for name, ids in [(f"학습 대상 {len(TARGET)}", TARGET), (f"E8-A 오답 {len(E8A_ERRORS)}", E8A_ERRORS)]:
        print(f"  {name}문항: E8-A 정답 {correct_in(e8a, ids)} -> E8-{CONFIG} 정답 {correct_in(h_final, ids)}")
    submit_ok = c2["b_only"] >= c2["a_only"]
    print(f"\n확인 지점 ②: {'통과 -> 9번 제출' if submit_ok else '미통과 -> 제출하지 않고 E8-A 유지'}")
    report["vs_E8A"] = c2
    report["target_fixed"] = {"E8A": correct_in(e8a, TARGET), f"E8{CONFIG}": correct_in(h_final, TARGET)}
    (DRIVE_RUN / "holdout_report.json").write_text(json.dumps(report, ensure_ascii=False, indent=2))
else:
    print("E8-A 결과가 없어 비교를 건너뜁니다:", E8A_PATH)

## 9. E8-3 — 제출

`SUBMIT = True`로 두고 실행하면 test 6,714문항에 같은 파이프라인을 적용합니다(약 1시간 40분).
Order4만 적용한 파일도 함께 만들어, LB에서 Zoom의 몫을 따로 볼 수 있게 합니다.

In [ ]:
SUBMIT = False

if not SUBMIT:
    print("SUBMIT = False: 제출을 건너뜁니다")
else:
    test_df = pd.read_csv(DATA_ROOT / "test.csv", keep_default_na=False)
    sample_df = pd.read_csv(DATA_ROOT / "sample_submission.csv", keep_default_na=False)
    assert list(test_df.id) == list(sample_df.id)
    test_rows = test_df.to_dict("records")
    t_o4 = tta.order4(tta.run_orders(ve, llm, proc, SAMPLING, test_rows, LOCAL_OUT / "test",
                                     DRIVE_RUN / "test", desc="test"))
    t_final, _ = zoom_blend(t_o4, test_rows, "test")
    for name, res in [("order4", t_o4), ("order4-zoom-blend", t_final)]:
        sub = sample_df[["id"]].merge(res[["id", "pred"]], on="id", how="left", validate="one_to_one")
        assert sub.pred.isin(ce.CHOICES).all() and list(sub.id) == list(sample_df.id)
        path = DRIVE_RUN / f"submission_qwen35-27b_lora-{CONFIG}_{name}.csv"
        sub.rename(columns={"pred": "answer"}).to_csv(path, index=False, encoding="utf-8")
        print(name, "예측 분포", sub.pred.value_counts(normalize=True).round(3).to_dict(), "->", path)
    print(f"Order4 대비 최종 답 변경 {int((t_o4.set_index('id').pred != t_final.set_index('id').pred).sum())}문항")

## 끝나면

Drive `output/exp_E8_tta_<구성>/`을 로컬 `output/`으로 받아 주세요. 순서별 예측(`holdout/<순서>/`, `holdout_zoom/<순서>/`),
`holdout_order4.csv`, `holdout_final.csv`, `holdout_report.json`, 제출 파일이 들어 있습니다.